# 실습 2a · 프롬프트 구체화: 역할 → 맥락 → 제약 → 출력 형식

발표자료의 4요소를 하나씩 더하며 같은 로그에 대한 답이 어떻게 바뀌는지 봅니다.

| 단계 | 더하는 것 | 기대하는 변화 |
|---|---|---|
| 0 | 없음 ("이 로그 위험해?") | 판정이 모호, 일반론 |
| 1 | 역할 | 보안 관점의 용어 · 어조 |
| 2 | 맥락 | 같은 출발지의 정찰 → 접속 흐름을 인식 |
| 3 | 제약 | 로그 근거 중심, 지어낸 내용 감소 |
| 4 | 출력 형식 | JSON으로 고정 → 프로그램이 바로 처리 |

로그는 sample_logs.txt의 3 · 8~12번째 줄(UNSW-NB15 원본 레코드)입니다. 내 PC의 Ollama로 보내므로 그대로 씁니다. 외부 LLM에 보내기 전 가리는 과정은 실습 2b에서 합니다.

### 접속 설정
`ENV`만 바꾸면 내 PC(Ollama)와 MonoRouter(원격)를 오갑니다. 실제 API 키는 이 파일에 저장한 채로 공유하지 않습니다.

In [13]:
from openai import OpenAI

ENV = "mono"          # "ollama" = 내 PC,  "mono" = MonoRouter(원격)
if ENV == "mono":
    client = OpenAI(base_url="https://monogpt.kr/api/monorouter/v1/openai/v1",
                    api_key="mr_openai_1KsRqTEI8s7Q8p1-XrN392il2EgpP16Z")
    MODEL = "gpt-5.5"
else:
    client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
    MODEL = "llama3.2:latest"
print(ENV, MODEL)

mono gpt-5.5


### ask(): 질문 하나 보내고 답 받기
답을 한꺼번에 기다리지 않고 만들어지는 대로 보여 줍니다(스트리밍). CPU로 도는 PC에서도 진행 중인지 바로 알 수 있습니다.

In [14]:
def ask(prompt, system=None, temperature=0.2, show=True):
    """질문을 보내고 답을 받아 돌려줌. show=True면 답이 만들어지는 대로 화면에 표시"""
    messages = [{"role": "user", "content": prompt}]
    if system:
        messages.insert(0, {"role": "system", "content": system})
    limit = {"max_tokens": 500} if ENV == "ollama" else {}   # 작은 모델이 끝없이 길게 답하는 것을 막음
    stream = client.chat.completions.create(model=MODEL, messages=messages,
                                            temperature=temperature, stream=True, **limit)
    answer = ""
    for chunk in stream:
        piece = chunk.choices[0].delta.content if chunk.choices else None
        if piece:
            answer += piece
            if show:
                print(piece, end="", flush=True)
    if show:
        print()
    return answer

In [15]:
import json, re

def parse_json(raw):
    """답에서 { ... } 부분만 꺼내 JSON으로 읽음. 없거나 깨졌으면 None"""
    m = re.search(r"\{.*\}", raw or "", re.DOTALL)
    try:
        return json.loads(m.group(0)) if m else None
    except json.JSONDecodeError:
        return None

## 로그와 채점 기준

In [16]:
import re, time

LINES = open("sample_logs.txt", encoding="utf-8").read().splitlines()
LOGS = "\n".join(LINES[i - 1] for i in [3, 8, 9, 10, 11, 12]) #골라낸 줄들을 다시 줄바꿈 문자(\n)로 이어붙여 하나의 텍스트 블록(LOGS)으로 만듦
print(LOGS)


2015-01-22 11:50:35Z [FLOW] src=59.166.0.7:12090 dst=149.171.126.3:53 proto=udp service=dns state=CON dur=0.001 spkts=2 dpkts=2 sbytes=146 dbytes=178 rate=3555.6 ct_dst_ltm=1 ct_src_dport_ltm=3
2015-01-22 11:50:49Z [FLOW] src=175.45.176.1:20549 dst=149.171.126.10:111 proto=udp service=- state=INT dur=0.000 spkts=2 dpkts=0 sbytes=168 dbytes=0 rate=2000.0 ct_dst_ltm=1 ct_src_dport_ltm=1
2015-01-22 11:50:52Z [FLOW] src=175.45.176.1:61099 dst=149.171.126.17:111 proto=tcp service=- state=FIN dur=0.633 spkts=10 dpkts=8 sbytes=564 dbytes=354 rate=28.5 ct_dst_ltm=1 ct_src_dport_ltm=1
2015-01-22 11:51:31Z [FLOW] src=175.45.176.1:38267 dst=149.171.126.10:80 proto=tcp service=http state=FIN dur=0.410 spkts=10 dpkts=8 sbytes=962 dbytes=354 rate=43.9 ct_dst_ltm=1 ct_src_dport_ltm=1
2015-01-22 11:52:18Z [FLOW] src=175.45.176.1:30074 dst=149.171.126.18:80 proto=tcp service=http state=FIN dur=0.248 spkts=10 dpkts=8 sbytes=846 dbytes=1008 rate=72.6 ct_dst_ltm=1 ct_src_dport_ltm=1
2015-01-22 11:52:21Z [

## 4요소

In [17]:
ROLE = "당신은 금융기관 SOC(보안관제) 분석가입니다."

CONTEXT = """로그 필드 뜻:
- 한 줄 = 네트워크 연결 1건. src/dst=출발지/목적지 IP:포트, service=서비스(-는 식별 안 됨)
- state=연결 상태 (FIN 정상 종료, INT 응답 없이 끝남, CON 연결 중), dur=지속시간(초)
- spkts/dpkts=송신/수신 패킷 수, sbytes/dbytes=송신/수신 바이트, rate=초당 패킷 수
- 149.171.126.x는 우리 서버 대역이다.
판단 기준: 한 출발지가 짧은 시간에 여러 서버의 같은 포트(예: 111 rpcbind)를 두드린 뒤
웹 · 메일 서비스에 접속해 평소와 다른 크기로 주고받으면 정찰 → 침투 시도 의심"""

CONSTRAINTS = """- 로그에 있는 내용만 근거로 쓴다. 없는 정보는 추측하지 않는다.
- 근거가 부족하면 판단 보류로 한다.
- 한 줄만 보고 단정하지 않고, 같은 출발지의 여러 줄을 함께 본다."""

OUTPUT_FORMAT = """JSON으로만 답한다. 다른 문장은 쓰지 않는다.
{"verdict": "위협 / 정상 / 판단 보류 중 하나",
 "threat_type": "DoS / Exploits / Reconnaissance / 없음 중 하나",
 "risk_score": 0~100 정수,
 "evidence": ["판단에 쓴 로그 내용"]}"""

PARTS = [("역할", ROLE), ("맥락", CONTEXT), ("제약", CONSTRAINTS), ("출력 형식", OUTPUT_FORMAT)]

## 단계별 실행
`run_stage(n)`은 앞의 n개 요소로 프롬프트를 만들어 보내고 결과를 기록합니다. 셀 하나에 한 단계씩 실행하며 답을 비교합니다.

In [18]:
results = {}

# 핵심 사실 검증용 정규식 딕셔너리 : 답이 로그의 핵심 사실 4개 중 몇 개를 근거로 언급하는지 셈 (단순 키워드 확인)
KEY_FACTS = {
    "같은 출발지": r"175\.45\.176\.1|같은 출발지|동일 출발지",
    "111번 포트 탐색": r"\b111\b|rpc|포트 (스캔|탐색)|INT",
    "여러 서버 대상": r"여러 (서버|대상|호스트)|다수 (서버|대상|호스트)",
    "pop3 대량 수신": r"pop3|129538|129,538|\b110\b",
}
def run_stage(n):
    if n == 0:
        prompt = f"이 로그 위험해?\n{LOGS}"    #샘플로그파일에서 골라낸 6줄 로그만 전달한다  
    else:
        prompt = "\n\n".join(f"[{name}]\n{text}" for name, text in PARTS[:n]) + f"\n\n[로그]\n{LOGS}" # PARTS에 역할, 맥락, 제약, 출력형식 전달
        print(f"--- 이번에 더한 요소: {PARTS[n - 1][0]} ---\n{PARTS[n - 1][1]}\n")
    print("--- 답 ---")
    start = time.time()
    answer = ask(prompt)      #챗 스트리밍 
    facts = [k for k, pat in KEY_FACTS.items() if re.search(pat, answer)]    #핵심 사실 검증용 정규식 딕셔너리 체크 
    d = parse_json(answer)
    is_json = bool(d) and d.get("verdict") in ("위협", "정상", "판단 보류")
    if is_json:
        verdict = d["verdict"]
    else:                                    # 문장 속 단어로 추정 → '?' 표시 (프로그램이 읽기 어려움)
        m = re.search(r"판단 보류|위협|정상", answer)
        verdict = m.group(0) + "?" if m else "없음"
    results[n] = (verdict, len(facts), len(answer), is_json, time.time() - start)
    print(f"\n--- 판정 {verdict} | 근거 {len(facts)}/4 {facts} | {len(answer)}자 | JSON {'예' if is_json else '아니오'} | {results[n][4]:.1f}초 ---")

In [19]:
run_stage(0)   # 나쁜 예

--- 답 ---
네, **전체적으로 위험/의심 로그로 보는 게 맞습니다.**  
특히 `175.45.176.1`에서 내부 대역으로 보이는 `149.171.126.x` 여러 호스트/서비스에 짧은 시간 동안 접근한 흔적이 있습니다.

## 요약 판단

| 로그 | 판단 | 이유 |
|---|---:|---|
| `59.166.0.7 -> 149.171.126.3:53 udp dns` | 낮음 | 일반적인 DNS 질의처럼 보임. 패킷/바이트도 정상 범위 |
| `175.45.176.1 -> 149.171.126.10:111 udp` | 높음 | `port 111/rpcbind` 접근. 응답 없음 `state=INT`라 스캔/탐색 가능성 |
| `175.45.176.1 -> 149.171.126.17:111 tcp` | 높음 | `rpcbind` TCP 연결 완료. 취약 서비스 탐색/공격 가능성 |
| `175.45.176.1 -> 149.171.126.10:80` | 중간~높음 | HTTP 접근. 단독이면 정상일 수 있으나 같은 출발지의 다중 서비스 접근과 함께 보면 의심 |
| `175.45.176.1 -> 149.171.126.18:80` | 중간~높음 | 위와 동일 |
| `175.45.176.1 -> 149.171.126.18:110 pop3` | 높음 | POP3 연결에서 `dbytes=129538`로 서버가 비교적 많은 데이터를 전송. 메일 수신, 계정 접근, 데이터 유출 가능성 확인 필요 |

## 특히 의심되는 점

### 1. 같은 외부 IP가 여러 내부 호스트/포트 접근

```text
src=175.45.176.1
dst=149.171.126.10:111
dst=149.171.126.17:111
dst=149.171.126.10:80
dst=149.171.126.18:80
dst=149.171.126.18:110
```

짧은 시간 안에 `111`, `80`, `110` 등 여러 서비스에 접근했습니다.  
이는 단순 사용자 트래픽보다는 **스캔

In [20]:
run_stage(1)   # + 역할

--- 이번에 더한 요소: 역할 ---
당신은 금융기관 SOC(보안관제) 분석가입니다.

--- 답 ---
## 1) 종합 판단

**의심 IP `175.45.176.1`가 내부 대역 `149.171.126.0/24`의 여러 호스트/서비스를 순차적으로 접속**하고 있습니다.  
특히 **TCP/UDP 111 RPC 서비스 탐색 후 HTTP 접속, POP3 110 접속에서 내부 서버 → 외부 IP 방향으로 약 129KB 데이터 전송**이 확인됩니다.

금융기관 SOC 관점에서는 단순 스캔을 넘어 **내부 메일/POP3 서비스 접근 및 데이터 유출 가능성**이 있어 **High 등급 이상으로 대응**하는 것이 적절합니다.

---

## 2) 주요 의심 행위

| 시간 UTC | 출발지 | 목적지 | 서비스 | 판단 |
|---|---|---|---|---|
| 11:50:49 | `175.45.176.1` | `149.171.126.10:111/udp` | RPC/Portmapper 추정 | UDP 111 탐색, 응답 없음 |
| 11:50:52 | `175.45.176.1` | `149.171.126.17:111/tcp` | RPC/Portmapper 추정 | TCP 111 접속 성공 후 종료 |
| 11:51:31 | `175.45.176.1` | `149.171.126.10:80` | HTTP | 웹 서비스 확인/접속 |
| 11:52:18 | `175.45.176.1` | `149.171.126.18:80` | HTTP | 다른 내부 호스트 웹 접속 |
| 11:52:21 | `175.45.176.1` | `149.171.126.18:110` | POP3 | 내부 서버에서 외부로 `129,538 bytes` 전송 |

---

## 3) 핵심 이상 징후

### 3.1 동일 외부 IP의 다중 내부 호스트 접근
`175.45.176.1`이 짧은 시간 동안 여러 내부 IP에 접근했습니다.

대상 내부 호스트:

- `149.171.126.10`
- `149.

In [12]:
run_stage(2)   # + 맥락

--- 이번에 더한 요소: 맥락 ---
로그 필드 뜻:
- 한 줄 = 네트워크 연결 1건. src/dst=출발지/목적지 IP:포트, service=서비스(-는 식별 안 됨)
- state=연결 상태 (FIN 정상 종료, INT 응답 없이 끝남, CON 연결 중), dur=지속시간(초)
- spkts/dpkts=송신/수신 패킷 수, sbytes/dbytes=송신/수신 바이트, rate=초당 패킷 수
- 149.171.126.x는 우리 서버 대역이다.
판단 기준: 한 출발지가 짧은 시간에 여러 서버의 같은 포트(예: 111 rpcbind)를 두드린 뒤
웹 · 메일 서비스에 접속해 평소와 다른 크기로 주고받으면 정찰 → 침투 시도 의심

--- 답 ---
판단: **침해 시도 의심 / 우선순위 높음**

## 핵심 요약

출발지 **175.45.176.1**이 짧은 시간 내 우리 서버 대역 **149.171.126.x**의 여러 서버를 대상으로 **111/rpcbind 포트 탐색**을 수행한 뒤, 곧바로 **HTTP(80)** 및 **POP3(110)** 서비스에 접속했습니다. 특히 POP3 접속에서 우리 서버가 외부로 **약 129KB**를 전송한 점이 이상합니다.

이는 주어진 판단 기준상 **정찰 → 웹/메일 서비스 접근 → 비정상 크기 데이터 송수신** 흐름에 부합합니다.

---

## 의심 흐름 타임라인

| 시간 | 출발지 | 목적지 | 내용 | 판단 |
|---|---|---|---|---|
| 11:50:49 | 175.45.176.1 | 149.171.126.10:111/udp | 응답 없음, INT | rpcbind 탐색 의심 |
| 11:50:52 | 175.45.176.1 | 149.171.126.17:111/tcp | 연결 성립 후 종료, FIN | rpcbind 서비스 확인 가능성 |
| 11:51:31 | 175.45.176.1 | 149.171.126.10:80 | HTTP 접속 | 정찰 후 웹 접근 |
| 11:52:18 | 175.

In [21]:
run_stage(3)   # + 제약

--- 이번에 더한 요소: 제약 ---
- 로그에 있는 내용만 근거로 쓴다. 없는 정보는 추측하지 않는다.
- 근거가 부족하면 판단 보류로 한다.
- 한 줄만 보고 단정하지 않고, 같은 출발지의 여러 줄을 함께 본다.

--- 답 ---
판단: **175.45.176.1 출발지에 대해 정찰 후 웹/메일 서비스 접속으로 이어진 의심 행위로 판단**합니다.  
다만, 로그만으로는 실제 침투 성공 여부나 “평소와 다른” 여부를 확정할 기준선이 없어 **침투 시도 의심 수준**입니다.

### 근거

| 시간 | 출발지 | 목적지 | 내용 |
|---|---|---|---|
| 11:50:49 | 175.45.176.1 | 149.171.126.10:111/udp | rpcbind 포트 111 접근, `INT`, 응답 패킷 없음 `dpkts=0` |
| 11:50:52 | 175.45.176.1 | 149.171.126.17:111/tcp | 다른 내부 서버의 동일 포트 111 접근, `FIN` |
| 11:51:31 | 175.45.176.1 | 149.171.126.10:80/tcp | 웹 HTTP 접속 |
| 11:52:18 | 175.45.176.1 | 149.171.126.18:80/tcp | 다른 내부 서버 웹 HTTP 접속 |
| 11:52:21 | 175.45.176.1 | 149.171.126.18:110/tcp | POP3 접속, 수신량 큼 `dbytes=129538`, `dpkts=116` |

### 분석

- `175.45.176.1`은 약 1분 30초 안에 내부 서버 대역 `149.171.126.x`의 여러 서버에 접근했습니다.
- 먼저 `149.171.126.10`, `149.171.126.17`의 **111 포트**에 접근했습니다.  
  - 111 포트는 rpcbind 계열 서비스로 볼 수 있으며, 여러 서버의 동일 포트를 짧은 시간에 확인한 점은 **서비스 확인 또는 정찰 행위**로 볼 수 있습니다.
- 이후 같은 출발지가 내부 웹 서

In [23]:
run_stage(4)   # + 출력 형식

--- 이번에 더한 요소: 출력 형식 ---
JSON으로만 답한다. 다른 문장은 쓰지 않는다.
{"verdict": "위협 / 정상 / 판단 보류 중 하나",
 "threat_type": "DoS / Exploits / Reconnaissance / 없음 중 하나",
 "risk_score": 0~100 정수,
 "evidence": ["판단에 쓴 로그 내용"]}

--- 답 ---
{"verdict":"위협","threat_type":"Reconnaissance","risk_score":80,"evidence":["175.45.176.1이 2015-01-22 11:50:49Z에 149.171.126.10:111/udp로 접속했으며 state=INT, dpkts=0, dbytes=0으로 응답 없이 종료됨","175.45.176.1이 2015-01-22 11:50:52Z에 149.171.126.17:111/tcp로 접속해 다른 내부 서버의 동일 포트 111을 연속적으로 접근함","같은 출발지 175.45.176.1이 이후 2015-01-22 11:51:31Z에 149.171.126.10:80 http, 11:52:18Z에 149.171.126.18:80 http로 접속함","같은 출발지 175.45.176.1이 2015-01-22 11:52:21Z에 149.171.126.18:110 pop3로 접속했고 dpkts=116, dbytes=129538로 큰 수신 바이트가 발생함","한 출발지가 짧은 시간에 여러 내부 서버의 111 포트를 접근한 뒤 웹 및 메일 서비스에 접속한 패턴이 관찰됨"]}

--- 판정 위협 | 근거 3/4 ['같은 출발지', '111번 포트 탐색', 'pop3 대량 수신'] | 572자 | JSON 예 | 7.2초 ---


## 단계별 비교

In [24]:
print(f"{'단계':<6}{'판정':<8}{'근거 수':>7}{'길이':>7}{'JSON':>6}{'시간(초)':>9}")
for n, (verdict, facts, length, is_json, sec) in sorted(results.items()):
    print(f"{n:<6}{verdict:<8}{facts:>5}/4{length:>7}{'예' if is_json else '-':>6}{sec:>9.1f}")

단계    판정         근거 수     길이  JSON    시간(초)
0     정상?         4/4   2151     -     16.3
1     정상?         4/4   4989     -     31.3
3     없음          4/4   1507     -     14.4
4     위협          3/4    572     예      7.2


### 관찰 포인트
- 단계 0은 일반론이 많고 판정이 문장 속에 섞여 프로그램이 읽기 어렵습니다(판정 뒤 '?').
- 다른 모델과 비교하려면 접속 설정의 MODEL을 "gemma3:1b"로 바꿔(미리 ollama pull) 다시 실행합니다.
- 단계 2(맥락) 뒤에 175.45.176.1이 111번 포트를 여러 서버에 두드린 뒤 웹 · pop3로 접속한 흐름을 하나로 묶는지 봅니다. 정답: 정상 1 · 정찰 2 · 침투 2 · DoS 1줄 (`../data/sample_logs_truth.csv`)
- 단계 3(제약) 뒤에 로그에 없는 내용(악성코드 이름, 공격 그룹)을 지어내는 문장이 줄어드는지 봅니다.
- 단계 4에서 JSON이 '예'가 되어야 실습 3 · 4처럼 여러 로그를 자동으로 처리할 수 있습니다.
- 답이 500토큰에서 끊기면 마지막 문장이 잘립니다. 형식이 없는 단계 0 · 1에서 자주 생깁니다.